# MACURA on an unstable quadrotor — a study guide

This notebook is a **guide**, not just a train/test script. It explains the
MACURA paper (*Trust the Model Where It Trusts Itself*, Frauenknecht et al.,
ICML 2024, [arXiv:2405.19014](https://arxiv.org/abs/2405.19014)), then reproduces
and stress-tests it on a **PyBullet quadrotor hover/takeoff** task, comparing the
four algorithms the paper studies: **MACURA, MBPO, M2AC, SAC**.

**How to read it.** Every code cell is preceded by prose saying *why* it exists
and *what to look for*. All heavy logic lives in the pure-function modules
(`envs/`, `models/`, `algorithms/`, `training/`, `viz/`); the notebook only
orchestrates, narrates, and plots. Set **`SMOKE = True`** (default) for a fast
top-to-bottom run; `False` for the full matrix.

Sections: **1** paper intro & paper-vs-project · **2** algorithm study · **3**
faster training setup · **4** train + save best model · **5** final evaluation +
videos · **6** results comparison & discussion.

## Setup — Drive, GPU, clone, install, config
Bootstrap (do not skip). Mount Drive, **require a GPU**, securely clone the
private repo, install deps via `setup_colab.sh`, and load the config.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MUJOCO_GL'] = 'egl'   # headless rendering (only used by the mujoco backend)
import torch
assert torch.cuda.is_available(), 'No GPU. Set Runtime -> Change runtime type -> GPU.'
DEVICE = 'cuda'
print('GPU OK:', torch.cuda.get_device_name(0))

In [ ]:
TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

Runs `setup_colab.sh` (clone + install + gym-pybullet-drones from source +
numpy repair). **One-time restart:** if a prior run corrupted numpy in this
kernel, run this once, then **Runtime → Restart session → Run all**.

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=macura-drone
curl -sH "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys
if '/content/macura-drone' not in sys.path:
    sys.path.insert(0, '/content/macura-drone')
import copy, json, yaml, numpy as np
import pybullet, gym_pybullet_drones, stable_baselines3, gymnasium
# pybullet has no __version__ attribute -> fall back to its API version int
PYB_VER = getattr(pybullet, '__version__', None) or pybullet.getAPIVersion()
print('versions: torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', PYB_VER,
      '| gym-pybullet-drones', getattr(gym_pybullet_drones, '__version__', '?'),
      '| gymnasium', gymnasium.__version__, '| numpy', np.__version__)

Load the config (single source of truth) and define `make_cfg`, which deep-merges
the optional **fast** speed profile and applies tiny `SMOKE` step counts. No
hyperparameters are hard-coded here — they all come from the YAML.

In [ ]:
with open('/content/macura-drone/configs/macura_drone.yaml') as f:
    CFG = yaml.safe_load(f)
DRIVE = CFG['experiment']['drive_root']
SMOKE = True   # <-- False for the full matrix (long)

def _deep_merge(base, over):
    for k, v in over.items():
        if isinstance(v, dict) and isinstance(base.get(k), dict):
            _deep_merge(base[k], v)
        else:
            base[k] = v
    return base

def make_cfg(smoke=False, fast=False, **env_overrides):
    c = copy.deepcopy(CFG)
    if fast:
        _deep_merge(c, c.get('profiles', {}).get('fast', {}))
    if smoke:
        c['experiment'].update(total_env_steps=1500, warmup_random_steps=300,
                               eval_every_steps=500, eval_episodes=2, seeds=[0])
        c['rollout']['num_rollouts'] = 100
        c['rollout']['freq_steps'] = 250
    c['env']['backend'] = 'pybullet'
    c['env']['mjcf_scene'] = '/content/mujoco_menagerie/skydio_x2/scene.xml'
    for k, v in env_overrides.items():
        c['env'][k] = v
    return c

cfg_full = make_cfg(smoke=SMOKE, fast=False)
cfg_fast = make_cfg(smoke=SMOKE, fast=True)
cfg = cfg_fast   # env study + training use the fast profile
print('SMOKE', SMOKE, '| steps', cfg['experiment']['total_env_steps'],
      '| seeds', cfg['experiment']['seeds'], '| backend', cfg['env']['backend'])

In [ ]:
from envs import drone_env
from viz import plots
from training import train as trainer
# probe the env so the gym-pybullet-drones API is confirmed before we depend on it
env, obs_dim, act_dim = drone_env.make_env(cfg['env'], seed=0, render=False)
obs0, _ = env.reset(seed=0)
print('obs_dim', obs_dim, '| act_dim', act_dim, '| obs[:7]', np.round(obs0[:7], 3))
print('HOVER_RPM', round(env.HOVER_RPM, 1), '| MAX_RPM', round(env.MAX_RPM, 1))

---
# Section 1 — The paper, and how this project differs

**The problem (Sec. 1).** Model-based RL trains the agent on *imagined* rollouts
from a learned dynamics model, which is sample-efficient — but model errors
**compound** the further you imagine, so the policy learns to exploit dynamics
that don't exist (*model exploitation*). MBPO (Janner 2019) grows the rollout
length on a fixed time schedule. MACURA instead asks *where* to trust the model.

**Concept → implementation:**

| Concept | Paper | This repo |
|---|---|---|
| Probabilistic ensemble | Sec 2.2, Eq 3–5 | `models/ensemble.py` |
| GJS uncertainty | Eq 15–19 | `compute_gjs` in `algorithms/macura.py` |
| Adaptive κ threshold | Eq 21 | `update_kappa`, `macura_rollout` |
| UTD gradient scaling | Eq 22 | `gradient_steps` in `algorithms/macura.py` |

The plot is an illustration of error compounding with horizon and where adaptive
truncation would cut it off.

In [ ]:
plots.plot_model_error_growth(save_path=f'{DRIVE}/plots/s1_error_growth.png');

## 1.1 The ensemble and GJS uncertainty
The ensemble (Eq. 3) predicts the next-state delta as a Gaussian per member.
Where members **agree** the model is trustworthy; where they **disagree** (far
from data) it is not. MACURA quantifies that disagreement with the **geometric
Jensen–Shannon divergence** (Eq. 15–19), which is closed-form for diagonal
Gaussians. Left: a real ensemble agreeing in-distribution and fanning out OOD.
Right: GJS rising with member spread.

In [ ]:
plots.plot_ensemble_toy_1d(cfg['ensemble'], device=DEVICE,
                           save_path=f'{DRIVE}/plots/s1_ensemble_toy.png')
plots.plot_gjs_vs_spread(save_path=f'{DRIVE}/plots/s1_gjs_vs_spread.png');

## 1.2 Adaptive truncation (Eq. 21) and UTD (Eq. 22)
κ is self-tuning: the running mean of `ξ ×` (the ζ-quantile of first-step
uncertainties); paper defaults `ζ=95%`, `T_max=10`. A rollout **continues while
`u_GJS < κ`, else truncates** (Algorithm 2). Because adaptive rollouts produce a
variable amount of data, MACURA scales SAC gradient steps with model-buffer
fullness (Eq. 22) — which is also a confound to control (Section 6).

In [ ]:
plots.plot_rollout_truncation_cartoon(save_path=f'{DRIVE}/plots/s1_truncation.png');

## 1.3 Noise types — pink vs white
Exploration noise matters. **White noise** is uncorrelated in time (jittery);
**pink noise** is `1/f` — temporally *correlated*, giving smoother, momentum-like
exploration. The paper uses **pink noise for MACURA only** (Sec. 6.2 / 7.3;
Eberhard et al. 2023), interacting deterministically for MBPO/M2AC. **This
project applies one scheme to all four** (`exploration.type` in the config) to
avoid handing MACURA an unfair exploration advantage. The plot shows both as a
time series and their power spectra — pink has the characteristic `1/f` slope.

In [ ]:
plots.plot_noise_comparison(save_path=f'{DRIVE}/plots/s1_noise.png');

## 1.4 Paper vs project — summary

| Aspect | Paper (MACURA) | This project |
|---|---|---|
| Backbone | mbrl-lib | **Stable-Baselines3 SAC** (mbrl-lib won't install on Py 3.12) |
| Ensemble | mbrl-lib PE | our PyTorch Gaussian ensemble (`models/ensemble.py`) |
| Domain | MuJoCo locomotion (stable) | **unstable PyBullet quadrotor** (errors compound fast) |
| Reward | known in theory / learned in practice | **known analytic** dense reward, shared by all four |
| Exploration | pink for MACURA only | **one scheme for all four** (fairness) |
| Goal | beat SoTA on benchmark | **reproduce the mechanism + transfer** to a hard domain |

A *mixed/negative* result on the drone is still a valid finding — the question is
whether the mechanism transfers, not whether we beat the paper's numbers.

---
# Section 2 — Algorithm study

All four share the **same SAC backbone** and (model-based ones) the **same
ensemble** — only the rollout strategy differs. That is what makes the comparison
attributable to the rollout strategy alone.

- **MACURA** — adaptive truncation: continue while `u_GJS < κ`, else stop
  (`algorithms/macura.py:macura_rollout`).
- **MBPO** — fixed truncated-linear schedule, no uncertainty
  (`algorithms/mbpo.py:mbpo_rollout`).
- **M2AC** — fixed length, then *mask* the least-trustworthy transitions + an
  uncertainty reward penalty (`algorithms/m2ac.py:m2ac_rollout`).
- **SAC** — model-free, no rollouts (`algorithms/sac.py:build_sac`).

The schematic contrasts rollout length vs training step for each strategy.

In [ ]:
plots.plot_rollout_strategy_schematic(save_path=f'{DRIVE}/plots/s2_strategy.png');

## 2.1 Why uncertainty is the lever
MACURA and M2AC both consume the GJS estimate but use it differently: MACURA sets
the rollout *length* (spatial/temporal truncation), M2AC *filters* fixed-length
rollouts. MBPO ignores uncertainty entirely. The illustration below ties the
uncertainty signal (member spread → GJS) back to the truncation decision.

In [ ]:
plots.plot_gjs_vs_spread(save_path=f'{DRIVE}/plots/s2_gjs.png')
plots.plot_rollout_truncation_cartoon(save_path=f'{DRIVE}/plots/s2_truncation.png');

---
# Section 3 — Faster training setup

**Speed profile.** `cfg_fast` (from `profiles.fast` in the YAML) trims the most
expensive knobs — fewer SAC updates per step (`gradient_steps_max`), fewer
ensemble epochs (`train_epochs_per_round`), fewer parallel rollouts
(`num_rollouts`), and less-frequent evaluation. This trades a little fidelity for
a much shorter wall-clock; use `cfg_full` for final numbers.

**Vectorization (no fidelity loss).** Two hot paths were de-looped:
`select_actions` now does one batched SB3 `predict` over the whole rollout batch
(was M Python calls), and `_store_model_transitions` does a single vectorized
bulk insert into the replay buffer (was row-by-row — tasks.md Phase 2). Same
results, fewer Python-loop overheads.

**Which noise, and why.** *Pink* exploration noise (config `exploration.type`)
is applied to **all four** algorithms for fairness. *Initialization* noise
(`env.init_*_noise`) is always on — episodes start near hover with small random
perturbations so competence is reachable in few steps. *Sensor/action* noise
(`env.noise`) is **off by default** and only enabled for the robustness study.

In [ ]:
print('fast vs full deltas:')
print('  gradient_steps_max:', cfg_fast['sac']['gradient_steps_max'], 'vs', cfg_full['sac']['gradient_steps_max'])
print('  train_epochs_per_round:', cfg_fast['ensemble']['train_epochs_per_round'], 'vs', cfg_full['ensemble']['train_epochs_per_round'])
print('  num_rollouts:', cfg_fast['rollout']['num_rollouts'], 'vs', cfg_full['rollout']['num_rollouts'])
print('  eval_every_steps:', cfg_fast['experiment']['eval_every_steps'], 'vs', cfg_full['experiment']['eval_every_steps'])
print('exploration:', cfg['exploration']['type'], '| sensor noise enabled:', cfg['env']['noise']['enabled'])

## 3.1 Robustness preview — clean vs noisy rollout
Enabling sensor + action noise (the robustness study) visibly degrades a rollout.
Compare these traces with the clean dynamics; later this is where an
uncertainty-aware method may degrade more gracefully.

In [ ]:
cfg_noisy = make_cfg(smoke=SMOKE, fast=True)
cfg_noisy['env']['noise'] = {'enabled': True, 'obs_noise_std': 0.05, 'action_noise_std': 0.05}
nenv, _, _ = drone_env.make_env(cfg_noisy['env'], seed=0, render=False)
rec_clean = plots.collect_rollout(env, policy='random', num_steps=150, seed=1)
rec_noisy = plots.collect_rollout(nenv, policy='random', num_steps=150, seed=1)
plots.plot_rollout_traces(rec_clean, title='Clean dynamics', save_path=f'{DRIVE}/plots/s3_clean.png')
plots.plot_rollout_traces(rec_noisy, title='With sensor+action noise', save_path=f'{DRIVE}/plots/s3_noisy.png');

---
# Section 4 — Train + save the best model to Drive

`trainer.train_one` runs the Dyna loop on the GPU and **does not evaluate every
step** — it evaluates periodically and **checkpoints only the best policy** to
`{DRIVE}/checkpoints/<algo>_seed<seed>_best.zip`, overwriting when a higher
return is reached. It returns the progress curves for plotting.

## 4.1 Smoke test — one short run validates the loop on PyBullet

In [ ]:
smoke_run = trainer.train_one('macura', make_cfg(smoke=True, fast=True), DRIVE, seed=99)
ks = [k for _, k in smoke_run['kappa']]
print('kappa finite:', bool(np.all(np.isfinite(ks))) if ks else 'n/a',
      '| best_return:', round(smoke_run['best_return'], 1),
      '| ckpt:', smoke_run['checkpoint'])

## 4.2 Full matrix — 4 algorithms × seeds (idempotent)
Skips any run whose log JSON already exists in Drive (resume-friendly). Uses the
**fast** profile; switch `cfg` to `cfg_full` for final numbers.

In [ ]:
runs = []
for algo in cfg['experiment']['algorithms']:
    for seed in cfg['experiment']['seeds']:
        lp = f'{DRIVE}/logs/{algo}_seed{seed}.json'
        if os.path.exists(lp):
            print('skip (exists):', algo, seed); runs.append(json.load(open(lp)))
        else:
            runs.append(trainer.train_one(algo, cfg, DRIVE, seed=seed))
print('collected', len(runs), 'runs')

## 4.3 Training-progress plots
Return vs real steps, crash rate, and the MACURA signatures (κ & rollout length,
GJS over training). With `SMOKE=True` these are short — they validate the
pipeline; run `SMOKE=False` for the real curves.

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/s4_sample_eff.png')
plots.plot_failure_rate(runs, save_path=f'{DRIVE}/plots/s4_failure.png')
mac = next((r for r in runs if r['algo'] == 'macura'), None)
if mac:
    plots.plot_rollout_depth(mac, save_path=f'{DRIVE}/plots/s4_rollout_depth.png')
    plots.plot_gjs_over_training(mac, save_path=f'{DRIVE}/plots/s4_gjs.png')

---
# Section 5 — Final evaluation with video recording

Now load the **best saved model** for every algorithm × seed from Drive and
evaluate it (no training here). `trainer.evaluate_best` returns clean eval
metrics; `trainer.record_best_videos` renders a deterministic episode per model
to `{DRIVE}/videos/<algo>_seed<seed>.mp4` (PyBullet's CPU camera is reliable on
Colab). Then we plot the evaluation figures.

In [ ]:
eval_metrics = trainer.evaluate_best(cfg, DRIVE, device=DEVICE)
import pandas as pd
display(pd.DataFrame(eval_metrics))

In [ ]:
video_paths = trainer.record_best_videos(cfg, DRIVE, device=DEVICE)
from IPython.display import Video, display
for name, path in video_paths.items():
    print(name, '->', path)
    display(Video(path, embed=True, width=440))

In [ ]:
# optional: download each saved video locally
from google.colab import files
for path in video_paths.values():
    files.download(path)

## 5.1 Evaluation figures
Sample-efficiency (mean ± std over seeds), final-policy quality, and per-seed
spaghetti (honest variance).

In [ ]:
plots.plot_sample_efficiency(runs, save_path=f'{DRIVE}/plots/s5_sample_eff.png')
plots.plot_final_quality(runs, save_path=f'{DRIVE}/plots/s5_final_quality.png')
plots.plot_sample_efficiency_spaghetti(runs, save_path=f'{DRIVE}/plots/s5_spaghetti.png');

---
# Section 6 — Results comparison & discussion

Read the results against the hypotheses (honestly — a mixed result on the
unstable drone is valid):
- **Does the model help?** MACURA / MBPO / M2AC vs **SAC** on sample efficiency.
- **Does adaptivity matter?** **MACURA vs MBPO** on stability / failure rate.
- **Truncation vs masking?** **MACURA vs M2AC** on return and stability.

The table summarizes best return + final failure rate per run; write the verdict
below it once `SMOKE=False` runs are in.

In [ ]:
rows = [{'algo': r['algo'], 'seed': r['seed'],
         'best_return': round(r.get('best_return', max(r['eval_return']) if r['eval_return'] else 0), 1),
         'final_fail_rate': round(r['eval_failure_rate'][-1], 2) if r['eval_failure_rate'] else None}
        for r in runs]
summary = pd.DataFrame(rows).groupby('algo').agg(
    best_return_mean=('best_return', 'mean'),
    best_return_std=('best_return', 'std'),
    fail_rate_mean=('final_fail_rate', 'mean')).reset_index()
display(summary)

## 6.1 Fairness audit & reproducibility footer
Confirms the controls held (one exploration scheme; UTD policy documented; MBPO
schedule is a real tunable schedule, not a strawman; GJS finite) and prints the
reproducibility metadata + Drive artifact paths.

In [ ]:
import hashlib
print('FAIRNESS AUDIT')
print('  exploration (all algos):', cfg['exploration']['type'])
print('  MACURA adaptive UTD:', cfg['rollout']['macura']['adaptive_gradient_steps'],
      '| MBPO/M2AC fixed UTD:', cfg['rollout']['mbpo']['fixed_gradient_steps'])
print('  MBPO schedule [min,max,start,end]:', cfg['rollout']['mbpo']['rollout_schedule'])
if mac and mac['kappa']:
    kv = [k for _, k in mac['kappa']]
    print('  kappa finite:', bool(np.all(np.isfinite(kv))))
if mac and mac['gjs']:
    print('  GJS finite:', bool(np.all(np.isfinite([g for _, g in mac['gjs']]))))
print('\nREPRODUCIBILITY')
print('  backend:', cfg['env']['backend'], '| SMOKE:', SMOKE,
      '| seeds:', cfg['experiment']['seeds'], '| steps:', cfg['experiment']['total_env_steps'])
print('  config md5:', hashlib.md5(json.dumps(CFG, sort_keys=True).encode()).hexdigest()[:8])
print('  versions: torch', torch.__version__, '| sb3', stable_baselines3.__version__,
      '| pybullet', PYB_VER, '| numpy', np.__version__)
for sub in ['plots', 'videos', 'logs', 'checkpoints']:
    d = f'{DRIVE}/{sub}'
    if os.path.isdir(d):
        print(f'  {sub}:', len(os.listdir(d)), 'files')